# Experiment: Cross-Method Oracle vs Baselines

Objective:
- Use the same six threshold-suite scenarios as the oracle-beta notebook.
- Compare `oracle_mcmcis`, `simple_mcmcis`, and `samc`.
- Reuse the saved simple/oracle MCMC-IS settings from the completed beta-oracle run.
- Track every method at dense checkpoints from `0.25M` through `5M` budget per run.

In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    _mcmc_eval_count,
    _steps_per_chain,
    build_beta_initialization,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_cross_method_saved_output,
    load_beta_sweep_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    plot_named_method_convergence,
    plot_named_method_max_budget,
    read_json,
    run_named_mcmc_checkpoint_study,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
    tune_samc_setup,
    write_json,
    write_jsonl,
    _effective_n_jobs,
    _iid_replicate_worker,
    _samc_replicate_worker,
    _try_make_process_pool,
)

pd.set_option("display.max_columns", 100)
project_root

## Configuration

This notebook reuses the saved simple/oracle initialization values from the beta notebook and does **not** run any pilot.  
Each method is evaluated with `20` independent one-chain runs, so the full `5M` is the actual chain budget and is not reduced by any initialization or tuning overhead.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
BETA_RESULTS_ROOT = project_root / "results" / "mcmcis_beta_notebook"


def latest_oracle_beta_run_dir(root: Path) -> Path:
    candidates = sorted(
        [
            path for path in Path(root).iterdir()
            if path.is_dir() and path.name.endswith("_oracle_beta_search")
        ]
    )
    if not candidates:
        raise FileNotFoundError(
            "No oracle-beta-search run found under results/mcmcis_beta_notebook. "
            "Run mcmcis_oracle_beta_search.ipynb first, or set BETA_RUN_DIR manually."
        )
    return candidates[-1]


BETA_RUN_DIR = latest_oracle_beta_run_dir(BETA_RESULTS_ROOT)
OUTPUT_ROOT = project_root / "results" / "cross_method_notebook"

SCENARIO_KEYS_OVERRIDE = [
    "gwas_additive_score_ultra_n100",
    "poisson_diffmeans_hep_ultra_n200",
    "gwas_additive_score_sig_n100",
    "poisson_diffmeans_hep_sig_n200",
    "gwas_additive_score_above_n100",
    "poisson_diffmeans_hep_above_n200",
]

MIN_TAIL_STATES = 2
BASE_SEED = 12_345
N_METHOD_RUNS = 20 if not FAST_MODE else 3
N_JOBS = min(6, os.cpu_count() or 1)
CHAIN_BUDGET = 5_000_000 if not FAST_MODE else 1_000_000
CHECKPOINT_STEP = 250_000 if not FAST_MODE else 250_000
ESTIMATION_POINTS = tuple(range(CHECKPOINT_STEP, CHAIN_BUDGET + CHECKPOINT_STEP, CHECKPOINT_STEP))

METHOD_ORDER = ["samc", "simple_mcmcis", "oracle_mcmcis"]
METHOD_LABELS = {
    "samc": "SAMC",
    "simple_mcmcis": "Simple-init MCMC-IS",
    "oracle_mcmcis": "Oracle MCMC-IS",
}
METHOD_COLORS = {
    "samc": "#4c8c77",
    "simple_mcmcis": "#c48a3a",
    "oracle_mcmcis": "#b04a5a",
}

cross_cfg = CrossMethodStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=N_METHOD_RUNS,
    base_seed=BASE_SEED,
    iid_density_samples=150_000 if not FAST_MODE else 20_000,
    min_tail_states=MIN_TAIL_STATES,
    n_jobs=N_JOBS,
)
mcmc_template_cfg = MCMCWorkflowConfig(
    pilot_samples=0,
    tune_steps=0,
    chains=1,
    burn_in_fraction=0.20,
    thin=1,
    estimate_variance=True,
    obm_batch_size=None,
    chain_n_jobs=1,
    tilt_mode="smooth_hinge",
    proposal_size=1,
    local_scan_enabled=False,
)
samc_base_cfg = SAMCWorkflowConfig(
    n_bins=100,
    t0=1_000.0,
    trace_every=200 if not FAST_MODE else 50,
    lambda_min_pilot=10_000 if not FAST_MODE else 1_000,
    proposal_size=0.1,
)

NOTEBOOK_CONFIG = {
    "FAST_MODE": FAST_MODE,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
    "BETA_RUN_DIR": str(BETA_RUN_DIR),
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "MIN_TAIL_STATES": MIN_TAIL_STATES,
    "BASE_SEED": BASE_SEED,
    "N_METHOD_RUNS": N_METHOD_RUNS,
    "N_JOBS": N_JOBS,
    "CHAIN_BUDGET": CHAIN_BUDGET,
    "CHECKPOINT_STEP": CHECKPOINT_STEP,
    "N_CHECKPOINTS": len(ESTIMATION_POINTS),
    "METHOD_ORDER": METHOD_ORDER,
    "USES_SAVED_INIT_ONLY": True,
    "COUNTS_INCLUDE_PILOT_BUDGET": False,
    "PLOT_X_SCALE": "linear",
    "CONVERGENCE_FIGURES": ["mean_estimate", "median_estimate"],
}

print(json.dumps(NOTEBOOK_CONFIG, indent=2))

## Notebook Helpers

In [ ]:
def proposal_size_for_scenario(scenario) -> int:
    setting_key = str(scenario.extra.get("application_setting_key", ""))
    if setting_key == "gwas_threshold_suite":
        return 2
    if setting_key == "hep_threshold_suite":
        return 6
    band = str(scenario.portfolio.get("sample_size_band", "medium"))
    if band == "small":
        return 1
    if band == "large":
        return 2
    return 1


def samc_cfg_for_scenario(scenario):
    return replace(
        samc_base_cfg,
        proposal_size=int(proposal_size_for_scenario(scenario)),
    )


def load_beta_reference(beta_run_dir: Path, scenario_key: str) -> dict:
    scenario_dir = Path(beta_run_dir) / scenario_key
    best_config = read_json(scenario_dir / "best_config.json")
    simple_summary = read_json(scenario_dir / "simple_init_summary.json")
    oracle_summary = read_json(scenario_dir / "oracle_best_trial.json")
    metadata = read_json(scenario_dir / "metadata.json")
    return {
        "scenario": scenario_key,
        "beta_run_dir": str(scenario_dir),
        "sigma_t": float(best_config["sigma_t"]),
        "reference_p0": float(best_config["reference_p0"]),
        "simple_reference_p0": float(best_config.get("simple_reference_p0", best_config["reference_p0"])),
        "canonical_threshold_p0": best_config.get("canonical_threshold_p0"),
        "simple_beta": float(best_config["beta_init"]),
        "simple_proposal_size": int(simple_summary["proposal_size"]),
        "oracle_beta": float(best_config["best_beta"]),
        "oracle_proposal_size": int(best_config["best_proposal_size"]),
        "oracle_trial_number": int(best_config["best_trial_number"]),
        "oracle_objective_value": float(best_config["best_objective_value"]),
        "selection_source": str(best_config.get("selection_source", "unknown")),
        "application_setting_key": metadata.get("application_setting_key"),
        "known_significance_threshold": metadata.get("known_significance_threshold"),
        "simple_summary": simple_summary,
        "oracle_summary": oracle_summary,
    }


def build_mcmc_config_specs(beta_reference: dict) -> list[dict]:
    return [
        {
            "label": "simple_mcmcis",
            "config_id": "simple_mcmcis",
            "beta": float(beta_reference["simple_beta"]),
            "proposal_size": int(beta_reference["simple_proposal_size"]),
            "source": "oracle_beta_search_simple",
        },
        {
            "label": "oracle_mcmcis",
            "config_id": f"oracle_trial_{int(beta_reference['oracle_trial_number']):02d}",
            "beta": float(beta_reference["oracle_beta"]),
            "proposal_size": int(beta_reference["oracle_proposal_size"]),
            "source": "oracle_beta_search_oracle",
        },
    ]


def run_parallel_worker_jobs(worker_fn, jobs: list[dict], *, n_jobs: int) -> list[dict]:
    n_workers = _effective_n_jobs(int(n_jobs), len(jobs))
    executor = _try_make_process_pool(n_workers) if n_workers > 1 else None
    rows: list[dict] = []
    if executor is None:
        for job in jobs:
            rows.extend(worker_fn(**job))
        return rows

    with executor:
        futures = [executor.submit(worker_fn, **job) for job in jobs]
        for future in futures:
            rows.extend(future.result())
    return rows


def run_samc_baseline(scenario, *, base_seed: int, samc_cfg) -> tuple[list[dict], dict]:
    checkpoints = tuple(int(v) for v in cross_cfg.estimation_points)
    samc_setup = tune_samc_setup(
        scenario.problem,
        samc_cfg,
        seed=int(base_seed + 50_000),
    )
    samc_jobs = [
        {
            "scenario_key": scenario.key,
            "scenario_display": scenario.description,
            "problem": scenario.problem,
            "exact_p": scenario.exact_p,
            "checkpoints": checkpoints,
            "samc_setup": samc_setup,
            "samc_cfg": samc_cfg,
            "rep": int(rep),
            "rep_seed": int(base_seed + 100_000 + 1_000 * rep),
        }
        for rep in range(int(cross_cfg.repeats))
    ]
    samc_rows = run_parallel_worker_jobs(_samc_replicate_worker, samc_jobs, n_jobs=int(cross_cfg.n_jobs))
    for row in samc_rows:
        row["label"] = "samc"

    return samc_rows, samc_setup


def save_oracle_cross_method_outputs(
    scenario,
    study: dict,
    *,
    output_dir: Path,
    beta_reference: dict,
    samc_cfg,
) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    plot_named_method_max_budget(
        study["records"],
        scenario_name=study["scenario_display"],
        scenario_key=study["scenario"],
        exact_p=float(study["exact_p"]),
        max_budget=max(int(v) for v in study["estimation_points"]),
        method_order=METHOD_ORDER,
        method_labels=METHOD_LABELS,
        method_colors=METHOD_COLORS,
        n_control=int(scenario.problem.n_control),
        n_treated=int(scenario.problem.n_treated),
        save_path=output_dir / "cross_method_max_budget.png",
    )
    plot_named_method_convergence(
        study["summary"],
        scenario_name=study["scenario_display"],
        scenario_key=study["scenario"],
        exact_p=float(study["exact_p"]),
        method_order=METHOD_ORDER,
        method_labels=METHOD_LABELS,
        method_colors=METHOD_COLORS,
        n_control=int(scenario.problem.n_control),
        n_treated=int(scenario.problem.n_treated),
        x_label="Budget per run",
        save_path=output_dir / "cross_method_convergence.png",
    )
    plot_named_method_convergence(
        study["summary"],
        scenario_name=study["scenario_display"],
        scenario_key=study["scenario"],
        exact_p=float(study["exact_p"]),
        method_order=METHOD_ORDER,
        method_labels=METHOD_LABELS,
        method_colors=METHOD_COLORS,
        n_control=int(scenario.problem.n_control),
        n_treated=int(scenario.problem.n_treated),
        x_label="Budget per run",
        estimate_field="median_estimate",
        estimate_title="Median estimate",
        estimate_ylabel=r"median $\hat{p}$",
        save_path=output_dir / "cross_method_convergence_median.png",
    )
    write_jsonl(output_dir / "run_records.jsonl", study["records"])
    write_json(output_dir / "summary.json", study["summary"])
    write_json(
        output_dir / "metadata.json",
        {
            "scenario": study["scenario"],
            "scenario_display": study["scenario_display"],
            "scenario_portfolio": study["scenario_portfolio"],
            "exact_p": study["exact_p"],
            "exact_method": study["exact_method"],
            "exact_tail_hits": study["exact_tail_hits"],
            "exact_n_perm": study["exact_n_perm"],
            "n_treated": int(scenario.problem.n_treated),
            "n_control": int(scenario.problem.n_control),
            "n_total": int(scenario.problem.n),
            "estimation_points": study["estimation_points"],
            "method_order": METHOD_ORDER,
            "method_labels": METHOD_LABELS,
            "method_colors": METHOD_COLORS,
            "x_label": "Budget per run",
            "x_scale": "linear",
            "convergence_plot_fields": {
                "mean": "mean_estimate",
                "median": "median_estimate",
            },
            "cross_config": cross_cfg,
            "mcmc_template_config": mcmc_template_cfg,
            "samc_config": samc_cfg,
            "beta_reference": beta_reference,
            "samc_setup": study["samc_setup"],
            "notebook_config": NOTEBOOK_CONFIG,
        },
    )


def regenerate_oracle_cross_method_plots(
    scenario_dir: Path,
    *,
    save_dir: Path | None = None,
) -> dict[str, Path]:
    saved = load_cross_method_saved_output(scenario_dir)
    metadata = saved["metadata"]
    save_dir = Path(save_dir) if save_dir is not None else Path(scenario_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    max_budget = max(int(v) for v in metadata["estimation_points"])
    out = {
        "cross_method_max_budget": save_dir / "cross_method_max_budget.png",
        "cross_method_convergence": save_dir / "cross_method_convergence.png",
        "cross_method_convergence_median": save_dir / "cross_method_convergence_median.png",
    }
    plot_named_method_max_budget(
        saved["records"],
        scenario_name=str(metadata["scenario_display"]),
        scenario_key=str(metadata["scenario"]),
        exact_p=float(metadata["exact_p"]),
        max_budget=max_budget,
        method_order=list(metadata.get("method_order", METHOD_ORDER)),
        method_labels=dict(METHOD_LABELS),
        method_colors=dict(METHOD_COLORS),
        n_control=int(metadata["n_control"]),
        n_treated=int(metadata["n_treated"]),
        save_path=out["cross_method_max_budget"],
    )
    plot_named_method_convergence(
        saved["summary"],
        scenario_name=str(metadata["scenario_display"]),
        scenario_key=str(metadata["scenario"]),
        exact_p=float(metadata["exact_p"]),
        method_order=list(metadata.get("method_order", METHOD_ORDER)),
        method_labels=dict(METHOD_LABELS),
        method_colors=dict(METHOD_COLORS),
        n_control=int(metadata["n_control"]),
        n_treated=int(metadata["n_treated"]),
        x_label=str(metadata.get("x_label", "Budget per run")),
        save_path=out["cross_method_convergence"],
    )
    plot_named_method_convergence(
        saved["summary"],
        scenario_name=str(metadata["scenario_display"]),
        scenario_key=str(metadata["scenario"]),
        exact_p=float(metadata["exact_p"]),
        method_order=list(metadata.get("method_order", METHOD_ORDER)),
        method_labels=dict(METHOD_LABELS),
        method_colors=dict(METHOD_COLORS),
        n_control=int(metadata["n_control"]),
        n_treated=int(metadata["n_treated"]),
        x_label=str(metadata.get("x_label", "Budget per run")),
        estimate_field="median_estimate",
        estimate_title="Median estimate",
        estimate_ylabel=r"median $\hat{p}$",
        save_path=out["cross_method_convergence_median"],
    )
    return out

## Load Scenarios and Oracle References

In [ ]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None,
    min_tail_states=MIN_TAIL_STATES,
)

beta_references = {
    scenario.key: load_beta_reference(BETA_RUN_DIR, scenario.key)
    for scenario in scenarios
}
run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "cross_method_oracle_compare") if SAVE_OUTPUTS else None

pd.DataFrame(
    [
        {
            "scenario": scenario.key,
            "exact_p": scenario.exact_p,
            "tail_hits": scenario.exact_tail_hits,
            "n_perm": scenario.exact_n_perm,
            "family": scenario.portfolio.get("family"),
            "rarity_band": scenario.portfolio.get("rarity_band"),
            "simple_beta": beta_references[scenario.key]["simple_beta"],
            "simple_prop": beta_references[scenario.key]["simple_proposal_size"],
            "oracle_beta": beta_references[scenario.key]["oracle_beta"],
            "oracle_prop": beta_references[scenario.key]["oracle_proposal_size"],
            "reference_p0": beta_references[scenario.key]["reference_p0"],
        }
        for scenario in scenarios
    ]
)

## Run Cross-Method Study

For each scenario:
- load the saved simple/oracle MCMC-IS settings from `BETA_RUN_DIR`,
- run `20` independent one-chain replicates for each method,
- save the article-facing max-budget plot plus both mean- and median-based convergence plots.

In [ ]:
cross_results = {}

for scenario_idx, scenario in enumerate(scenarios):
    beta_reference = beta_references[scenario.key]
    scenario_seed = int(BASE_SEED + 1_000_000 * (scenario_idx + 1))
    scenario_samc_cfg = samc_cfg_for_scenario(scenario)
    mcmc_specs = build_mcmc_config_specs(beta_reference)

    print(f"Running {scenario.key} | exact p={scenario.exact_p:.3e}")
    print(json.dumps({
        "scenario": scenario.key,
        "n_method_runs": int(cross_cfg.repeats),
        "n_jobs": int(cross_cfg.n_jobs),
        "chain_budget": CHAIN_BUDGET,
        "first_checkpoints": [int(v) for v in ESTIMATION_POINTS[:4]],
        "last_checkpoint": int(ESTIMATION_POINTS[-1]),
        "simple_beta": beta_reference["simple_beta"],
        "simple_proposal_size": beta_reference["simple_proposal_size"],
        "oracle_beta": beta_reference["oracle_beta"],
        "oracle_proposal_size": beta_reference["oracle_proposal_size"],
        "sigma_t": beta_reference["sigma_t"],
        "samc_proposal_size": scenario_samc_cfg.proposal_size,
    }, indent=2))

    mcmc_study = run_named_mcmc_checkpoint_study(
        scenario.problem,
        scenario.exact_p,
        config_specs=mcmc_specs,
        sigma_t=float(beta_reference["sigma_t"]),
        estimation_points=tuple(int(v) for v in cross_cfg.estimation_points),
        repeats=int(cross_cfg.repeats),
        base_seed=scenario_seed,
        template_cfg=mcmc_template_cfg,
        n_jobs=int(cross_cfg.n_jobs),
    )
    samc_rows, samc_setup = run_samc_baseline(
        scenario,
        base_seed=scenario_seed + 300_000,
        samc_cfg=scenario_samc_cfg,
    )

    records = list(mcmc_study["records"]) + list(samc_rows)
    records = sorted(
        records,
        key=lambda row: (
            str(row.get("label", row.get("method", ""))),
            int(row["replicate"]),
            int(row["checkpoint"]),
        ),
    )
    summary = summarize_records(records, group_fields=("checkpoint", "label"))

    study = {
        "scenario": scenario.key,
        "scenario_display": scenario.description,
        "scenario_portfolio": dict(scenario.portfolio),
        "exact_p": float(scenario.exact_p),
        "exact_method": scenario.exact_method,
        "exact_tail_hits": int(scenario.exact_tail_hits),
        "exact_n_perm": int(scenario.exact_n_perm),
        "estimation_points": [int(v) for v in cross_cfg.estimation_points],
        "records": records,
        "summary": summary,
        "beta_reference": beta_reference,
        "samc_setup": {
            "lambda_min": float(samc_setup["lambda_min"]),
            "bin_edges": samc_setup["bin_edges"],
        },
    }
    cross_results[scenario.key] = study

    if SAVE_OUTPUTS and run_dir is not None:
        save_oracle_cross_method_outputs(
            scenario,
            study,
            output_dir=run_dir / scenario.key,
            beta_reference=beta_reference,
            samc_cfg=scenario_samc_cfg,
        )

    summary_df = pd.DataFrame(summary).sort_values(["checkpoint", "label"])
    display(
        summary_df[
            [
                "checkpoint",
                "label",
                "mean_estimate",
                "median_estimate",
                "rmse",
                "mean_abs_log10_error",
                "mean_eval_excl_tuning",
                "mean_q_tilt_tail_share",
                "mean_ess",
                "mean_acceptance_rate",
                "mean_zero_rate",
                "mean_samc_max_rel_freq_error",
            ]
        ]
    )

family_rows = []
for scenario in scenarios:
    meta = scenario.portfolio
    for row in cross_results[scenario.key]["summary"]:
        family_rows.append(
            {
                "scenario": scenario.key,
                "family": meta.get("family"),
                "rarity_band": meta.get("rarity_band"),
                "difficulty": meta.get("expected_difficulty"),
                **row,
            }
        )

family_df = pd.DataFrame(family_rows)
display(
    family_df.groupby(["family", "rarity_band", "label", "checkpoint"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_estimate=("mean_estimate", "mean"),
        mean_abs_log10_error=("mean_abs_log10_error", "mean"),
        mean_q_tilt_tail_share=("mean_q_tilt_tail_share", "mean"),
        mean_ess=("mean_ess", "mean"),
        mean_acceptance_rate=("mean_acceptance_rate", "mean"),
    )
    .sort_values(["family", "rarity_band", "checkpoint", "label"])
)

## Review Saved Figures

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    for scenario in scenarios:
        scenario_dir = run_dir / scenario.key
        print(f"\n{scenario.key}")
        display(Image(filename=str(scenario_dir / "cross_method_max_budget.png")))
        display(Image(filename=str(scenario_dir / "cross_method_convergence.png")))
        display(Image(filename=str(scenario_dir / "cross_method_convergence_median.png")))
else:
    print("SAVE_OUTPUTS=False, so no saved figures to display.")

## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_SCENARIO_DIR = None
# # Example:
# # RELOAD_SCENARIO_DIR = project_root / "results" / "cross_method_notebook" / "20260429_120000_cross_method_oracle_compare" / "gwas_additive_score_sig_n100"

# if RELOAD_SCENARIO_DIR is not None:
#     saved = load_cross_method_saved_output(RELOAD_SCENARIO_DIR)
#     print(json.dumps({
#         "scenario": saved["metadata"]["scenario"],
#         "exact_p": saved["metadata"]["exact_p"],
#         "methods": saved["metadata"]["method_order"],
#     }, indent=2))
#     regen = regenerate_oracle_cross_method_plots(RELOAD_SCENARIO_DIR)
#     for name, path in regen.items():
#         print(name, path)
#         display(Image(filename=str(path)))
# else:
#     print("Set RELOAD_SCENARIO_DIR to a saved scenario directory to regenerate the three article-facing plots.")